In [3]:
!pip install catboost
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"  # allows multiple prints from a cell

# install all packages and dependencies so that the code cell runs without errors.

import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os, plotly.express as px
# import xgboost as xgb, lightgbm, re, tensorflow as tf, tensorflow.keras as keras
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
import sklearn.preprocessing # trasnformers
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.neural_network import MLPRegressor  # SKLearn's MLP is optimised for CPU (and doesn't use GPU)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error as mse, r2_score
# other imports or implementation (for example, Robust Regressor with interpretable match-case statements)

np.set_printoptions(linewidth=10000, precision=2, edgeitems=20, suppress=True)
pd.set_option('display.max_colwidth', 100, 'display.max_columns', 10, 'display.width', 1000, 'display.max_rows', 8)

In [4]:
train_sample = pd.read_csv('train_sample.csv')
train_sample

,start_point,end_point,time_of_day,day_of_week,traffic_condition,...,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,...,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,...,high,NaN,1,1.081668,27.489129
2,Central Jakarta (Jakarta Pusat),East Jakarta (Jakarta Timur),morning,Thursday,NaN,...,low,NaN,2,1.192379,27.228978
3,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Friday,10.0,...,high,fog,1,0.833348,33.943970
...,...,...,...,...,...,...,...,...,...,...,...
39996,North Jakarta (Jakarta Utara),South Jakarta (Jakarta Selatan),morning,Friday,9.0,...,NaN,fog,1,0.914077,55.668274
39997,North Jakarta (Jakarta Utara),South Jakarta (Jakarta Selatan),evening,Friday,5.0,...,low,fog,1,0.943298,60.548584
39998,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Thursday,10.0,...,NaN,rain,1,1.185288,23.996753
39999,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),night,Saturday,NaN,...,NaN,rain,2,0.873630,13.841914


In [5]:
test_sample = pd.read_csv('test_sample.csv')
test_sample

,start_point,end_point,time_of_day,day_of_week,traffic_condition,...,vehicle_density,population_density,weather,public_transport_availability,historical_delay_factor
0,West Jakarta (Jakarta Barat),East Jakarta (Jakarta Timur),morning,Saturday,5.0,...,medium,NaN,NaN,2,1.126429
1,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),evening,Saturday,NaN,...,low,medium,fog,2,1.121015
2,Central Jakarta (Jakarta Pusat),South Jakarta (Jakarta Selatan),morning,Friday,9.0,...,high,NaN,rain,1,1.109638
3,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),night,Wednesday,5.0,...,low,NaN,storm,1,0.842474
...,...,...,...,...,...,...,...,...,...,...,...
2996,Central Jakarta (Jakarta Pusat),East Jakarta (Jakarta Timur),day,Monday,NaN,...,high,medium,NaN,2,0.941115
2997,South Jakarta (Jakarta Selatan),East Jakarta (Jakarta Timur),day,Saturday,5.0,...,medium,high,fog,2,0.973585
2998,North Jakarta (Jakarta Utara),West Jakarta (Jakarta Barat),day,Tuesday,NaN,...,low,medium,clear,0,0.815122
2999,North Jakarta (Jakarta Utara),West Jakarta (Jakarta Barat),night,Friday,9.0,...,NaN,high,rain,0,0.873886


In [6]:
fill_mode = lambda col: col.fillna(col.mode())
train_sample = train_sample.fillna({k: v[0] for k, v in train_sample.mode().to_dict().items()})
test_sample = test_sample.fillna({k: v[0] for k, v in train_sample.mode().to_dict().items()})

In [7]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["start_end_point"] = start + " " + end
test_sample["start_end_point"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [8]:
y_train = train_sample['travel_time']
X_train = train_sample.drop('travel_time', axis=1)
X_test = test_sample
# etc.
# your code here

In [9]:
cat_features = X_train.select_dtypes(
    include=['str', 'object']
).columns.tolist()

print("Categorical features:", cat_features)

Categorical features: ['start_point', 'end_point', 'time_of_day', 'day_of_week', 'vehicle_density', 'population_density', 'weather', 'start_end_point']


In [11]:
from sklearn.model_selection import KFold, cross_validate

In [12]:
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error

for depth in [3, 4, 5, 6, 7, 8]:
    model = CatBoostRegressor(
        iterations=1000,
        depth=depth,
        learning_rate=0.05,
        loss_function="RMSE",
        cat_features=tuple(cat_features),
        verbose=False,
        random_seed=38
    )

    cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
        return_train_score=True
    )

    # print("CV RMSE:", -scores["test_score"].mean())
    print(f"Depth: {depth}")
    print('R²:', scores['test_r2'].mean())
    print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
    print('MSE:', -scores['test_neg_mean_squared_error'].mean())
    print('---')


Depth: 3
R²: 0.9150390711372968
MAE: 2.7008620208569813
MSE: 19.36539486249539
---
Depth: 4
R²: 0.9158832882258086
MAE: 2.6792416282861202
MSE: 19.172347308833814
---
Depth: 5
R²: 0.9160002777761569
MAE: 2.6723140209397633
MSE: 19.146870693771326
---
Depth: 6
R²: 0.916101821299239
MAE: 2.667693472343641
MSE: 19.12394549859058
---
Depth: 7
R²: 0.9161734328608901
MAE: 2.6660527071988858
MSE: 19.10776132726971
---
Depth: 8
R²: 0.9161129798263751
MAE: 2.6644784038245133
MSE: 19.12200975935448
---


In [ ]:
y_train_hat = model.predict(X_train)
mse_lr = mse(y_train_hat, y_train)
r2_lr = r2_score(y_train_hat, y_train)
mse_lr, r2_lr
# also: RMSE will estimate the error on the dataset in the original units

(19.41198954491261, 0.9025646518175551)